In [6]:
import requests
import pandas as pd
import re
from datetime import datetime

client_id = "tUpceN2tI9JBcwm05yBW"
client_secret = "OW_SC2bJMD"

url = "https://openapi.naver.com/v1/search/news.json"

headers = {
    "X-Naver-Client-Id": client_id,
    "X-Naver-Client-Secret": client_secret
}

keywords = [
    "숏폼",
    "쇼츠",
    "릴스",
    "틱톡",
    "숏폼드라마"
]

def clean_text(text):
    if not text:
        return ""
    
    text = re.sub("<.*?>", "", text)
    text = text.replace("&quot;", '"')
    text = text.replace("&amp;", "&")
    text = text.replace("&lt;", "<")
    text = text.replace("&gt;", ">")
    return text.strip()

rows = []

for keyword in keywords:
    params = {
        "query": keyword,
        "display": 100,
        "start": 1,
        "sort": "date"
    }

    response = requests.get(url, headers=headers, params=params)

    if response.status_code != 200:
        print("API 요청 실패:", keyword)
        print(response.status_code)
        print(response.text)
        continue

    data = response.json()

    for item in data.get("items", []):
        rows.append({
            "keyword": keyword,
            "title": clean_text(item.get("title", "")),
            "description": clean_text(item.get("description", "")),
            "originallink": item.get("originallink", ""),
            "link": item.get("link", ""),
            "pub_date": item.get("pubDate", ""),
            "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

df = pd.DataFrame(rows)

df.insert(0, "index", range(1, len(df) + 1))

df.to_csv(
    "naver_news_shortform.csv",
    index=False,
    encoding="utf-8-sig"
)

print(df.head())
print("뉴스 데이터 저장 완료")

   index keyword                                           title  \
0      1      숏폼     이준, 근육질 걸그룹 댄스에 유재석 폭소..'캐치 캐치' 이어 아일릿도 ...   
1      2      숏폼  [K-컬처가 여는 스마트관광의 미래 (6)] K-콘텐츠의 힘, 영화·드라마·팝...   
2      3      숏폼        [이대화의 함께 들어요] [40] 빨라진 K팝 노래, 배경엔 숏폼이 있다   
3      4      숏폼            "생명의 길 사계절 담으세요"… 농어촌공사, 어도 사진 공모 예고   
4      5      숏폼                 농협상호금융, 대학생 홍보단 ‘NH콕서포터즈’ 5기 모집   

                                         description  \
0  앞서 이준은 유튜브 '워크맨'에서 치어리더로 변신해 가수 최예나의 '캐치캐치' 댄스...   
1  틱톡 숏폼 영상, 유튜브 브이로그, 인스타그램 릴스와 같은 플랫폼이 관광객을 또 다...   
2  요즘 유행의 키를 쥔 숏폼 플랫폼이 속도 높은 음악들을 선호하면서 자연스럽게 흐름이...   
3  올해 공모전은 사진과 짧은 영상(숏폼) 등 2개 부문으로 나눠 운영된다. 사진 부문...   
4  카드뉴스와 숏폼 영상, SNS 콘텐츠 등을 직접 기획·제작하며 디지털 금융 서비스를...   

                                        originallink  \
0  https://www.starnewskorea.com/broadcast-show/2...   
1  https://www.news2day.co.kr/article/20260513500162   
2  https://www.chosun.com/opinion/specialist_colu...   
3  https://www.newscj.com/news

In [5]:
import requests
from bs4 import BeautifulSoup
import time
import re
from datetime import datetime
import pandas as pd
from urllib.parse import quote

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36"
}

queries = ["숏폼", "쇼츠", "숏폼드라마", "클립"]

ds = "2021.01.01"
de = "2026.05.14"
nso = "so:r,p:from20210101to20260514,a:all"

def clean_text(text):
    text = re.sub("<.*?>", "", text)
    return text.strip()

rows = []

for query in queries:
    print(f"{query} 수집 시작")

    encoded_query = quote(query)

    url = (
        "https://search.naver.com/search.naver"
        f"?where=news"
        f"&query={encoded_query}"
        f"&sm=tab_opt"
        f"&sort=0"
        f"&photo=0"
        f"&field=0"
        f"&pd=3"
        f"&ds={ds}"
        f"&de={de}"
        f"&nso={nso}"
    )

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    links = soup.select("a[href]")

    news_links = []

    for a in links:
        href = a.get("href", "")
        title = a.get_text(strip=True)

        if not title:
            continue

        if "news.naver.com" in href or "n.news.naver.com" in href:
            news_links.append((title, href))

    # 중복 제거
    news_links = list(dict.fromkeys(news_links))

    print(f"{query} 검색 결과 수:", len(news_links))

    for title, link in news_links:
        content = ""

        try:
            article_res = requests.get(link, headers=headers, timeout=5)
            article_soup = BeautifulSoup(article_res.text, "html.parser")

            content_tag = (
                article_soup.select_one("#dic_area") or
                article_soup.select_one("#articeBody") or
                article_soup.select_one("#articleBodyContents") or
                article_soup.select_one("article")
            )

            content = clean_text(content_tag.get_text(" ", strip=True)) if content_tag else ""

        except:
            content = ""

        rows.append({
            "keyword": query,
            "title": title,
            "content": content,
            "date": "",
            "link": link,
            "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

        time.sleep(0.2)

    print(f"{query} 수집 완료")

df = pd.DataFrame(rows)
df.insert(0, "index", range(1, len(df) + 1))

df.to_csv("naver_news_multi_5years.csv", index=False, encoding="utf-8-sig")

print(df.head())
print("총 수집 기사 수:", len(df))
print("뉴스 데이터 저장 완료")

숏폼 수집 시작
숏폼 검색 결과 수: 15
숏폼 수집 완료
쇼츠 수집 시작
쇼츠 검색 결과 수: 6
쇼츠 수집 완료
숏폼드라마 수집 시작
숏폼드라마 검색 결과 수: 5
숏폼드라마 수집 완료
클립 수집 시작
클립 검색 결과 수: 4
클립 수집 완료
   index keyword                                              title  \
0      1      숏폼  언론사 선정언론사가 선정한 주요기사 혹은 심층기획 기사입니다.네이버 메인에서 보고 ...   
1      2      숏폼                                              네이버뉴스   
2      3      숏폼                                              네이버뉴스   
3      4      숏폼                                              네이버뉴스   
4      5      숏폼                                              네이버뉴스   

                                             content date  \
0                                                           
1  이나은·최보민·윤현석·김도아 등 출연 요리 게임을 소재로 한 숏폼 드라마가 베일을 ...        
2  이나은·최보민 등 출연해 게임 IP 확장 및 신규 레스토랑 머쉬룸 가든 업데이트 비...        
3  강원랜드, 예방 메시지 발굴·확산 활동 진행 [정선=뉴시스] 강원랜드 청소년 도박예...        
4  숏폼·포스터·웹툰 분야서 총 12개 작품 시상 강원랜드 청소년 도박예방 콘텐츠 공모...        

                                                link         collect